# RevalExo external-validation pipeline

This notebook creates the frozen RevalExo external window set. RevalExo is never used for training, normalization, calibration, threshold selection, or model selection.

## Fixed signal contract

- Xsens order: Pelvis, Right Upper Leg, Right Lower Leg, Right Foot, Left Upper Leg, Left Lower Leg, Left Foot.
- Project output order: LB ← Pelvis; LF ← Left Foot; RF ← Right Foot.
- Use gravity-inclusive `acceleration` (m/s² converted to g), not gravity-removed `free_acceleration`: the internal magnitude model includes gravity.
- Convert gyroscope rad/s to degrees/s, resample to 100 Hz, and create 5-second windows with a 2.5-second hop only within annotated level-ground walking.


In [1]:
from pathlib import Path
import ast

import h5py
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'revalexo'
OUT = PROJECT_ROOT / 'data' / 'processed'
TARGET_HZ, WINDOW, HOP = 100.0, 500, 250
STANDARD_GRAVITY, RAD_TO_DEG = 9.80665, 180.0 / np.pi
XSENS_ORDER = ['Pelvis', 'Right Upper Leg', 'Right Lower Leg', 'Right Foot', 'Left Upper Leg', 'Left Lower Leg', 'Left Foot']

audit_rows, windows, window_rows = [], [], []
for folder in sorted(RAW_ROOT.glob('raw_full_part*/raw_full/Subject*')):
    group = folder.name.rsplit('_', 1)[-1]
    if group not in {'HC', 'ST'}:
        continue
    annotations = pd.read_csv(folder / 'annotations.csv')
    walking = annotations.loc[annotations['Task'].str.contains('level ground walking', case=False, na=False)]
    with h5py.File(folder / 'mvn-analyze.hdf5', 'r') as handle:
        trackers = handle['mvn-analyze/xsens-motion-trackers']
        acceleration = trackers['acceleration']
        gyroscope = trackers['gyroscope']
        headings = ast.literal_eval(''.join(list(acceleration.attrs['Data headings'])))
        acceleration_unit = acceleration.attrs['Units']
        gyroscope_unit = gyroscope.attrs['Units']
        assert headings == XSENS_ORDER and acceleration.shape[1] == 7 and gyroscope.shape[1] == 7
        time_s = trackers['process_time_s'][:].ravel()
        source_hz = 1.0 / np.median(np.diff(time_s))
        acceleration_g = np.asarray(acceleration[:, [0, 3, 6], :], dtype=np.float32) / STANDARD_GRAVITY
        gyroscope_deg_s = np.asarray(gyroscope[:, [0, 3, 6], :], dtype=np.float32) * RAD_TO_DEG
        # Canonical output order: LB (Pelvis), LF (Left Foot), RF (Right Foot).
        signal = np.concatenate([acceleration_g[:, 0], gyroscope_deg_s[:, 0], acceleration_g[:, 2], gyroscope_deg_s[:, 2], acceleration_g[:, 1], gyroscope_deg_s[:, 1]], axis=1)
        target_time_s = np.arange(time_s[0], time_s[-1], 1.0 / TARGET_HZ)
        resampled = np.column_stack([np.interp(target_time_s, time_s, signal[:, column]) for column in range(signal.shape[1])]).astype(np.float32)
    audit_rows.append({'subject': folder.name, 'group': group, 'walking_segments': len(walking), 'source_hz': source_hz, 'samples': len(time_s), 'duration_s': time_s[-1] - time_s[0], 'acceleration_field': 'acceleration', 'acceleration_unit': acceleration_unit, 'gyroscope_unit_metadata': gyroscope_unit})
    for _, segment in walking.iterrows():
        start = max(0, int(np.searchsorted(target_time_s, float(segment['Start_toa_s']))))
        stop = min(len(target_time_s), int(np.searchsorted(target_time_s, float(segment['End_toa_s']))))
        for window_start in range(start, stop - WINDOW + 1, HOP):
            window_id = len(windows)
            windows.append(resampled[window_start:window_start + WINDOW])
            window_rows.append({'window_id': window_id, 'subject': folder.name, 'group': group, 'window_start_index': window_start, 'window_seconds': WINDOW / TARGET_HZ, 'source_hz': source_hz})

audit = pd.DataFrame(audit_rows)
window_array = np.asarray(windows, dtype=np.float32)
window_metadata = pd.DataFrame(window_rows)
assert window_array.shape[1:] == (WINDOW, 18) and np.isfinite(window_array).all()
assert set(window_metadata['group']) == {'HC', 'ST'}
OUT.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUT / 'revalexo_external_validation_audit.csv', index=False)
np.save(OUT / 'revalexo_external_windows_float32.npy', window_array)
window_metadata.to_csv(OUT / 'revalexo_external_window_metadata.csv', index=False)

display(audit.groupby('group').agg(subjects=('subject', 'nunique'), walking_segments=('walking_segments', 'sum'), median_source_hz=('source_hz', 'median')))
display(window_metadata.groupby('group').agg(subjects=('subject', 'nunique'), windows=('window_id', 'size')))
print('Gravity-inclusive acceleration field:', audit['acceleration_field'].unique().tolist())
print('Window array:', window_array.shape)


,subjects,walking_segments,median_source_hz
group,,,
HC,7,1358,62.760796
ST,10,844,63.162625


,subjects,windows
group,,
HC,7,754
ST,10,1474


Gravity-inclusive acceleration field: ['acceleration']
Window array: (2228, 500, 18)
